In [ ]:
from pinoco import ODEEquation, SimulatedTrajectoryDataset, SubtrajectoryDataset, SubtrajectoryView, PINNTrainDataset, TorchODEResidual, make_pinn_dataloader

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset

from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

from collections import defaultdict

from solis_nn import SOLIS, get_global_signal_stats, get_global_time_scale, MultitrajectoryIPINN, SimpleIPINN, save_model_package, load_checkpoint
from solis_training import train_epoch, train_epoch_ipinn, train_epoch_simple_ipinn
from solis_eval import eval_model

ImportError: cannot import name 'LPVParamNetFiLM' from 'solis_nn' (/home/murat/Documents/mfm/solis/solis_nn.py)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Equation
# Duffing Oscillator Equation
eq_gt = ODEEquation(
    eqs=[
        "Eq(D(y,2), -delta*D(y,1) - alpha*y(t) - beta*y(t)**3 + u)"
    ],
    params={"delta": 0.2, "alpha": -1.0, "beta": 1.0, "gamma": 0.3, "omega": 1.2, },
    dependent_variables=["y"],
    exogenous_functions=["u"],
)

def make_exo_duffing(
    kind: str,
    amp=1.0,
    freqs=(0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50),
    noise_seed=0,
):
    kind = kind.lower()

    if kind == "multitone":
        return {
            "u": lambda t: 1.0 * torch.cos(0.5 * t) + 0.7 * torch.sin(2.3 * t)
        }

    if kind == "step":
        return {
            "u": lambda t: torch.full_like(t, float(amp))
        }

    if kind == "multisine":
        gen = torch.Generator(device="cpu")
        gen.manual_seed(int(noise_seed))
        freqs_t = torch.tensor(list(freqs), dtype=torch.float32)
        phases = 2.0 * torch.pi * torch.rand((len(freqs_t),), generator=gen)
        A = float(amp) / max(1, len(freqs_t)) ** 0.5  # keep RMS sane

        def u(t):
            tt = t.reshape(-1)
            s = 0.0
            for fi, ph in zip(freqs_t, phases):
                s = s + torch.sin(2.0 * torch.pi * fi * tt + ph)
            return (A * s).reshape_as(t)

        return {"u": u}

    raise ValueError(f"Unknown kind: {kind}")


# 2. Initial Conditions
N_traj_multitone = 3
N_traj_step = 2
N_traj_multisine = 3

y0_set_multitone = (torch.rand(N_traj_multitone, 2) - 0.5) * 5.0
y0_set_multitone[:, 1] = (torch.rand(N_traj_multitone) - 0.5) * 4.0

y0_set_step = (torch.rand(N_traj_step, 2) - 0.5) * 5.0
y0_set_step[:, 1] = (torch.rand(N_traj_step) - 0.5) * 4.0

y0_set_multisine = (torch.rand(N_traj_multisine, 2) - 0.5) * 5.0
y0_set_multisine[:, 1] = (torch.rand(N_traj_multisine) - 0.5) * 4.0

Tf = 16.0
NT = 1024

COMMON_SIM_KW = dict(
    t0=0.0,
    tf=Tf,
    T=NT,
    backend="torch",
    method="rk4",
    device="cpu",
    dtype=torch.float32,
    keep_graph=False,
)

# 3. Simulation 
gt_traj_multitone = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multitone,
    y0_set=y0_set_multitone,
    exogenous_torch=make_exo_duffing(kind="multitone"),
    **COMMON_SIM_KW,
)
subtraj_multitone = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_multitone, subseq_len=256)
)

gt_traj_step = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_step,
    y0_set=y0_set_step, 
    exogenous_torch=make_exo_duffing(kind="step"),
    **COMMON_SIM_KW,
)
subtraj_step = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_step, subseq_len=256)
)

gt_traj_multisine = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multisine,
    y0_set=y0_set_multisine,
    exogenous_torch=make_exo_duffing(kind="multisine", amp=1.0, noise_seed=0),
    **COMMON_SIM_KW,
)
subtraj_multisine = SubtrajectoryView(
    SubtrajectoryDataset(gt_traj_multisine, subseq_len=256)
)

# 5. Training sets (one per scenario, then concat)
pinn_multitone = PINNTrainDataset(
    subtraj_multitone,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_step = PINNTrainDataset(
    subtraj_step,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_multisine = PINNTrainDataset(
    subtraj_multisine,
    collocation_frac=1.0,
    num_data_per_traj=30,
    Tf=Tf,
    disjoint_sets=False,
    importance_y_weight=5,
)

pinn_train_dataset = ConcatDataset([pinn_multitone, pinn_step, pinn_multisine])

In [ ]:
loader = make_pinn_dataloader(
    pinn_train_dataset,
    n_traj_samples=8,
    n_data_samples=NT,
    seed=123,
    shuffle_trajs=False
)

## Simple Visualizations

In [ ]:
plt.figure(figsize=(10,6))
for b in loader:
    print(b["y0"].shape)
    print(b["t_col"].shape)
    print(b["y_data"].shape)
    print(b["t_data"].shape)
    print(b["traj_id"])

    for traj_id in range(b["y0"].shape[0]):
        plt.plot(b["t_col"][traj_id].numpy(), np.zeros_like((b["t_col"][traj_id].numpy())))
        plt.scatter(b["t_data"][traj_id].numpy(), b["y_data"][traj_id,:,0].numpy(), color='r',s=2)
        plt.scatter(b["t_data"][traj_id].numpy(), b["y_data"][traj_id,:,1].numpy(), color='g',s=2)
        plt.plot(b["t_col"][traj_id,0].numpy(), b["y0"][traj_id,:,0].numpy(), color='r', marker='o', markersize=8)
        plt.plot(b["t_col"][traj_id,0].numpy(), b["y0"][traj_id,:,1].numpy(), color='g', marker='o', markersize=8)
        plt.plot(b["t_col"][traj_id].numpy(), b["exo_col"]["u"][traj_id].numpy(), linestyle='--')

    
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

# Pick one trajectory and its sampled training set
dset = gt_traj_multitone[0]
pdset = pinn_train_dataset[0]

# Continuous ODE solution
ax.plot(dset["t"].cpu(), dset["y"][:,0].cpu(),
        label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

ax.plot(dset["t"].cpu(), dset["y"][:,1].cpu(),
        label="y(t) Solution", color="C0", linewidth=2, alpha=0.6)

ax.plot(dset["t"].cpu(), dset["exo"]["u"].cpu(),
        label="u(t)", color="C4", linewidth=2, alpha=0.6)

# Data points
ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,0].cpu(),
           label="Data Points", color="C1", marker="o", edgecolors="k")

ax.scatter(pdset["t_data"].cpu(), pdset["y_data"][:,1].cpu(),
           label="Data Points", color="C1", marker="o", edgecolors="k")

# Initial condition
ax.scatter(pdset["t_col"][0].cpu(), pdset["y0"][0].cpu(),
           label="Initial Value", color="C2", marker="s", s=80, edgecolors="k")

# Collocation points (y=0 just for visualization)
ax.scatter(pdset["t_col"].cpu(),
           torch.zeros_like(pdset["t_col"]).cpu(),
           label="Collocation Points", color="C3", marker="x")

# Styling
ax.set_xlabel("Time $t$")
ax.set_ylabel("$y(t)$")
ax.set_title(f"Trajectory")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend()

plt.tight_layout()
plt.show()

## Training

In [ ]:
global_t_min, global_t_scale = get_global_time_scale(pinn_train_dataset)
global_t_min = torch.tensor(global_t_min)
global_t_scale = torch.tensor(global_t_scale)

stats = get_global_signal_stats(pinn_train_dataset)

In [ ]:
# 2nd order hypothesis
plant_eq_y  = "Eq(D(y,1), v)"
plant_eq_dy = "Eq(D(v,1), -k*y - d*v + g*u)"

eq_pred = ODEEquation(
    eqs = [
        plant_eq_y,
        plant_eq_dy,
    ],  
    params={}, 
    dependent_variables=["y", "v"],
    exogenous_functions=["u", "k", "d", "g"],           
)

### solis Training

In [ ]:
cfg_solis = dict(
        x_dim=4, use_u_in_sol_net=True,
        include_abs_time=False, use_relative_time=True,
        use_moe=True, poly_order=3, ensure_positive_wn=False,
        include_abs=True, include_cross=True, include_energy=True,
        sol_net_layers=4, sol_net_hidden_dim=256, param_net_hidden_dim=256,
        use_fourier_time=True, include_u_in_params=True,
        traj_emb_dim=0, num_trajectories=len(pinn_train_dataset),
        use_input_normalization=True,
        context_hidden_dim=64, context_dim=32,
)

solis = SOLIS(**cfg_solis)

# solis.set_time_bounds(global_t_min.item(), (global_t_min + global_t_scale).item())
# model.set_norm_stats(
#     y_mean=stats["y_mean"],
#     y_std=stats["y_std"],
#     v_mean=stats["v_mean"],
#     v_std=stats["v_std"],
#     u_mean=stats["u_mean"],
#     u_std=stats["u_std"],
# )

plant_params = list(solis.y_net.parameters())
if getattr(solis, "traj_emb", None) is not None:
    plant_params += list(solis.traj_emb.parameters())

if solis.use_moe:
    param_params = (
        list(solis.param_feat.parameters()) +
        list(solis.gating_net.parameters()) +
        [p for exp in solis.experts for p in exp.parameters()]
    )
else:
    param_params = list(solis.param_feat.parameters()) + list(solis.param_head.parameters())


use_velocity_data = True
use_velocity_ic   = True

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_plant  = torch.optim.AdamW(plant_params, lr=5e-4, weight_decay=5e-5)
optimizer_params = torch.optim.AdamW(param_params, lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_plant  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_plant, mode="min", factor=0.5, patience=50)
scheduler_params = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_params, mode="min", factor=0.5, patience=50)

# Tensorboard
# now_str = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
# writer = SummaryWriter(log_dir=f"runs/solis-Phint-2nd{now_str}", flush_secs=3)
writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 5_000

# Define Phase Durations
phase1_epochs = 50  
phase2_epochs = 50  
epoch_period = phase1_epochs + phase2_epochs 

phint_cutoff_ratio = 0.1

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    phase_epoch = (epoch-1) % epoch_period

    # --------------------------------------
    # 1. Determine Phase & Hyperparameters
    # --------------------------------------
    if phase_epoch < phase1_epochs:
        current_phase = 1
        phase_desc = "Phase 1: Solution Learning"
        
    elif phase_epoch < (phase1_epochs + phase2_epochs):
        current_phase = 2
        phase_desc = "Phase 2: Parameter Identification"
    
    change_phase = not (current_phase == prev_phase)
    hint_decay = max(0.0, 1.0 - epoch / (epochs * phint_cutoff_ratio))
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch(
        model=solis,
        residual=residual,
        eq=eq_pred,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer_plant=optimizer_plant,
        optimizer_params=optimizer_params,
        criterion=criterion,
        phase=current_phase,  
        change_phase=change_phase,
        scheduler_plant=scheduler_plant,
        scheduler_params=scheduler_params,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
        lambda_gate_reg=0.01,
        lambda_phy_hint=1 * hint_decay,
        lambda_tv=1,
        H_horizon=10,
        lambda_step=0.5,
        random_window="medium"
    )

    prev_phase = current_phase

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    # Update tqdm status bar
    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"{phase_desc}")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

### IPINN-M Training

In [ ]:
cfg_ipinn_m = dict(
        x_dim=4, use_u_in_sol_net=True,
        include_abs_time=False, use_relative_time=True,
        use_moe=True, poly_order=3, ensure_positive_wn=False,
        include_abs=True, include_cross=True, include_energy=True,
        sol_net_layers=4, sol_net_hidden_dim=256,
        use_fourier_time=True, include_u_in_params=True,
        traj_emb_dim=0, num_trajectories=len(pinn_train_dataset),
        use_input_normalization=True,
        context_hidden_dim=64, context_dim=32,
)

ipinn_m = MultitrajectoryIPINN(**cfg_ipinn_m)

use_velocity_data = True
use_velocity_ic   = True

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_ipinn_m  = torch.optim.AdamW(ipinn_m.parameters(), lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_ipinn_m  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ipinn_m, mode="min", factor=0.5, patience=50)

writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 3_000

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch_ipinn(
        model=ipinn_m,
        residual=residual,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer=optimizer_ipinn_m,
        criterion=criterion,
        scheduler=scheduler_ipinn_m,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
    )

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"[Training]")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

### IPINN

In [ ]:
cfg_simple = dict(
        x_dim=4, use_u_in_sol_net=True,
        include_abs_time=False, use_relative_time=True,
        include_abs=True, include_cross=True, include_energy=True,
        sol_net_layers=4, sol_net_hidden_dim=256,
        use_fourier_time=True, include_u_in_params=True,
        use_input_normalization=True,
)

simple_ipinn = SimpleIPINN(**cfg_simple)

use_velocity_data = True
use_velocity_ic   = True

if use_velocity_data:
    print("Using position and velocity data for training.")
    data_dim = 2
else:
    print("Using position-only data for training.")
    data_dim = 1

if use_velocity_ic:
    print("Using position and velocity in initial conditions.")
    ic_dim = 2
else:
    print("Using position-only in initial conditions.")
    ic_dim = 1

# Optimizers 
optimizer_ipinn_simple  = torch.optim.AdamW(simple_ipinn.parameters(), lr=5e-4, weight_decay=5e-5)

# Losses and schedulers
criterion = nn.MSELoss()
residual = TorchODEResidual(eq_pred)
scheduler_ipinn_simple  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ipinn_simple, mode="min", factor=0.5, patience=50)

writer = None

In [ ]:
# ==========================================
# Configuration
# ==========================================
epochs = 3_000

# Track parameters
param_list = defaultdict(list)

# Progress bar
pbar = tqdm(range(1,epochs+1), desc="Initializing", ncols=120, dynamic_ncols=True)

prev_phase = 1

for epoch in pbar:
    
    # --------------------------------------
    # 2. Run Training Step
    # --------------------------------------
    step_info = train_epoch_simple_ipinn(
        model=simple_ipinn,
        residual=residual,
        loader=loader,
        ic_dim=ic_dim,
        data_dim=data_dim,
        optimizer=optimizer_ipinn_simple,
        criterion=criterion,
        scheduler=scheduler_ipinn_simple,
        lambda_data=10.0,
        lambda_ic=100.0,
        lambda_phys=10.,        
    )

    # --------------------------------------
    # 3. Logging
    # --------------------------------------
    # Log losses to TensorBoard
    if writer is not None:
        for key, value in step_info.items():
            writer.add_scalar(f"losses/{key}", value, epoch)
        writer.add_scalar("training/phase", current_phase, epoch)

    physics_loss = step_info['physics_loss']
    data_loss = step_info['data_loss']

    pbar.set_description(f"[Training]")
    pbar.set_postfix({
        "Data Loss": f"{data_loss:.2e}",
        "Physics Loss" : f"{physics_loss:.2e}",
        "IC": f"{step_info['ic_loss']:.1e}"
    })

Save

In [ ]:
# --- Save them ---
save_dir = "./models/duffing"

# Assuming your model instances are named: model_solis, model_multi, model_simple
save_model_package(solis, cfg_solis, f"{save_dir}/solis.pth")
save_model_package(ipinn_m,  cfg_ipinn_m,  f"{save_dir}/ipinn_m.pth")
save_model_package(simple_ipinn, cfg_simple, f"{save_dir}/simple_ipinn.pth")

## Plots

### Training Set Solution & Parameter Field Learning

In [ ]:
import importlib
import solis_eval 
importlib.reload(solis_eval)
from solis_eval import eval_model, plot_surrogate_rollout, plot_phase_comparison

eval_model({"solis" : solis, 
            "IPINN-M": ipinn_m, 
            "IPINN" : simple_ipinn
            }, 
          ConcatDataset([subtraj_multisine, subtraj_step, subtraj_multitone]), 
          pinn_train_dataset, convert_params=False, plot_params=False,
          save_path="./figures/duffing/duffing_solution.pdf",
          break_loop=True,
           )

### Surrogate Rollout
Test data

In [ ]:
# 2. Initial Conditions
N_traj_multitone = 1
N_traj_step = 1
N_traj_multisine = 1

y0_set_multitone_test = (torch.rand(N_traj_multitone, 2) - 0.5) * 5.0
y0_set_multitone_test[:, 1] = (torch.rand(N_traj_multitone) - 0.5) * 4.0

y0_set_step_test = (torch.rand(N_traj_step, 2) - 0.5) * 5.0
y0_set_step_test[:, 1] = (torch.rand(N_traj_step) - 0.5) * 4.0

y0_set_multisine_test = (torch.rand(N_traj_multisine, 2) - 0.5) * 5.0
y0_set_multisine_test[:, 1] = (torch.rand(N_traj_multisine) - 0.5) * 4.0

Tf = 16.0
NT = 1024

COMMON_SIM_KW = dict(
    t0=0.0,
    tf=Tf,
    T=NT,
    backend="torch",
    method="rk4",
    device="cpu",
    dtype=torch.float32,
    keep_graph=False,
)

test_traj_multitone = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multitone,
    y0_set=y0_set_multitone_test,
    exogenous_torch=make_exo_duffing(kind="multitone"),
    **COMMON_SIM_KW,
)

test_traj_step = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_step,
    y0_set=y0_set_step_test,  
    exogenous_torch=make_exo_duffing(kind="step"),
    **COMMON_SIM_KW,
)

test_traj_multisine = SimulatedTrajectoryDataset(
    ode_equation=eq_gt,
    n_trajectories=N_traj_multisine,
    y0_set=y0_set_multisine_test,
    exogenous_torch=make_exo_duffing(kind="multisine", amp=1.0, noise_seed=0),
    **COMMON_SIM_KW,
)

test_traj = ConcatDataset([test_traj_multitone, test_traj_step, test_traj_multisine])

In [ ]:
plot_surrogate_rollout({
    "solis" : solis, 
    "IPINN-M": ipinn_m, 
    "IPINN" : simple_ipinn
    }, 
    eq_pred, 
    test_traj,
    0, convert_params=False, plot_params=False,
    use_running_average_v=False,use_running_average_y=True,blend_alpha=0.0,
    save_path="./figures/duffing/duffing_surrogate.pdf"
    )

### State-space Vector Fields

In [ ]:
# Assuming your model and equations are ready
metrics = plot_phase_comparison(
    model=solis,
    eq_gt=eq_gt, 
    eq_pred=eq_pred,
    trajs=[traj["y"] for traj in ConcatDataset([subtraj_multisine, subtraj_step, subtraj_multitone])], 
    y_range=(-2.5, 2.5), 
    v_range=(-3, 3), 
    u0=0.0, # Plot autonomous dynamics (u=0)
    save_path="figures/duffing/duffing_phase_solis.pdf"
)
print(metrics)